# 08 · Bằng chứng Big Data: phễu dữ liệu, pandas vs Spark, khả năng mở rộng (phụ trách: Hùng)

**Đề:** *"distributed feature-engineering pipeline"*. Mã nguồn: `data_spark/scaling.py`, `deploy/spark/`.
Kết quả thật xuất hiện sau khi chạy `make bench` trên EC2.

In [ ]:
# --- Thiết lập chung ---------------------------------------------------------------
import json, os, sys
from pathlib import Path
import pandas as pd, numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

REAL = False          # True trên EC2 sau khi đã chạy `make trust TASK=...` với dữ liệu thật
TASK = "p1"           # p1 | p2 | p3

def load_results(task=TASK):
    p = ROOT / ("results/trust" if REAL else "artifacts/demo_results") / task / "results.json"
    if not p.exists():
        raise FileNotFoundError(f"Chưa có {p}. Chạy `python tools/make_demo_bundles.py` (demo) hoặc `make trust TASK={task}` (EC2).")
    return json.loads(p.read_text(encoding="utf-8"))

VERIFIED = json.loads((ROOT / "results/data/verified_v1.json").read_text(encoding="utf-8"))
print("Nguồn:", "DỮ LIỆU THẬT" if REAL else "DEMO (dữ liệu giả — số liệu không có ý nghĩa y khoa)")

## 1. Benchmark cũ (v1) và vì sao phải làm lại

In [ ]:
b = VERIFIED['benchmark_v1']; print(b['limitation']); pd.DataFrame(b['seconds'], index=['1%', '5%', '10%']).T

## 2. Kết quả mới (sau `make bench`)

In [ ]:
bench = ROOT / "results/benchmark"
for name in ("funnel.json", "pandas_full_load.json"):
    p = bench / name
    print(name, "→", json.loads(p.read_text(encoding="utf-8")) if p.exists() else "chưa chạy")
if (bench / "scaling.csv").exists():
    s = pd.read_csv(bench / "scaling.csv"); s = s[s["ok"]]
    t = s.groupby(["mode", "cores"])["seconds"].median().unstack(0); t["speedup_strong"] = t["strong"].iloc[0] / t["strong"]
    display(t)
else:
    print("Chưa có scaling.csv — chạy: make bench-strong bench-weak bench-ablation bench-report")

## Câu hỏi tự kiểm tra
1. Vì sao file `.csv.gz` chỉ đọc được bằng 1 task? Bước nào trong pipeline giải quyết?
2. Strong scaling và weak scaling khác nhau thế nào? E(n) giảm có phải là lỗi không (gợi ý: Amdahl)?
3. Broadcast join dùng ở đâu, tại sao? Ablation `no_auto_broadcast` cho thấy gì?
4. Chạy nhiều core trên MỘT máy khác cụm nhiều máy thế nào? (`deploy/spark/cluster.sh`)